In [ ]:
from pathlib import Path
import sys

src_dir = next(p for p in [Path.cwd(), *Path.cwd().parents]
               if (p / "common.py").is_file() and (p / "feature_preference").is_dir())
sys.path.insert(0, str(src_dir))


In [ ]:
import torch
import torch.nn as nn
from tqdm.auto import tqdm
import numpy as np
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
import random

from common import *
import yaml
import matplotlib.colors as mcolors
import pickle

import library

In [ ]:
device = torch.device("cuda")
sns.set_theme(style="whitegrid", font="cmr10", font_scale=1.9)
plt.rcParams["axes.formatter.use_mathtext"] = True
plt.rcParams.update({"pdf.fonttype": 42, "ps.fonttype": 42})
colors = sns.color_palette()
colors

default plot setting

In [ ]:
with open("../linear/config.yaml", 'r', encoding='utf-8') as f:
    configs_linear = yaml.safe_load(f)
with open("../deep/config.yaml", 'r', encoding='utf-8') as f:
    configs_deep = yaml.safe_load(f)
with open("../mlp/config.yaml", 'r', encoding='utf-8') as f:
    configs_mlp = yaml.safe_load(f)

config = configs_mlp
loss_type = "BarlowTwins_leakyrelu_log"

In [ ]:
def sample(batch, sizes, device, mul=0.01):
    x_out = torch.zeros((batch, 0), device=device)
    for size in sizes:
        rand = torch.rand((batch, 1), device=device).round() * 2 - 1
        x_out = torch.cat([x_out, rand.repeat(1, size)], dim=-1)
    epsilon1 = mul * torch.randn_like(x_out)
    epsilon2 = mul * torch.randn_like(x_out)
    return x_out + epsilon1, x_out + epsilon2

sample makes [+1, -1] datas

In [ ]:
def create_segment_vectors(sizes):
    total_length = sum(sizes)
    cumsum = torch.cumsum(torch.tensor(sizes), dim=-1)

    result = torch.zeros((len(sizes), total_length))

    for i in range(len(sizes)):
        start_idx = cumsum[i - 1] if i > 0 else 0
        end_idx = cumsum[i]

        result[i, start_idx:end_idx] = 1

    return result

In [ ]:
batch = config["batch"]
sizes = config["sizes"]
if "sizes" in config[loss_type]:
    sizes = config[loss_type]["sizes"]
s_in = sum(sizes)
s_out = len(sizes)

In [ ]:
sample_data = sample(batch, sizes, device, mul=0.01)
gamma = torch.bmm(sample_data[0].unsqueeze(dim=2), sample_data[1].unsqueeze(dim=1))  # b x n x n
gamma = gamma + gamma.permute(0, 2, 1)
gamma = torch.mean(gamma, dim=0).squeeze() / 2  # n x n
sns.heatmap(gamma.cpu())
plt.show()
gamma_l, gamma_v = torch.linalg.eig(gamma)
gamma_l = gamma_l.real.cpu()
gamma_v = gamma_v.real.cpu()
print(gamma_l, gamma_v)

In [ ]:
def train(loss_type, slope=None, layers=None):
    set_seed(config[loss_type]["seed"])

    mid_size = config["mid_layer_size"]
    total_layer = layers
    mid_layer = []
    if "leakyrelu" in loss_type:
         def activation_types():
            return nn.LeakyReLU(negative_slope=float(slope), inplace=True)
    else:
         def activation_types():
            return nn.ReLU(inplace=True)
    for _ in range(total_layer - 2):
        mid_layer.append(nn.Linear(mid_size, mid_size, bias=False, device=device))
        mid_layer.append(activation_types())
    encoder = nn.Sequential(nn.Linear(s_in, mid_size, bias=False, device=device), activation_types(), *mid_layer,
                            nn.Linear(mid_size, s_out, bias=False, device=device))
    scaling = float(config[loss_type]["scaling"])

    encoder.train()

    eiglist = [[] for i in range(s_out)]

    if config["network"] != "linear":
        scale_weights(encoder, scaling ** (1 / config["layer"]))
    else:
        scale_weights(encoder, scaling)

    w = get_total_weight(encoder).detach().clone()
    u0, s0, v0T = torch.linalg.svd(w)
    v0 = v0T.T

    x0, x1 = sample(batch, sizes, device, mul=0.01)
    z0, z1 = torch.chunk(encoder(torch.cat([x0, x1], dim=0)), 2, dim=0)
    c, loss = lossFn(z0, z1)
    c = c.detach()
    v = reversed(torch.linalg.eigvalsh(c))
    for i in range(len(v)):
        eiglist[i].append(v[i].cpu().item())
    return eiglist[0][0]

In [ ]:
e = create_segment_vectors(sizes).to(device)

slopes = np.linspace(0, 1, 1000+1)
layers = range(2, 9)
eigList = [[] for _ in layers]
for l in range(len(layers)):
    for slope in tqdm(slopes):
        lossFn = BTLoss(use_batchnorm=False, in_feature=s_out, lambd=float("0")).to(device)
        eigList[l].append(train(loss_type, slope=slope, layers=l))

In [ ]:
fig, axs = plt.subplots(1, 1, figsize=(12, 10))
print(len(eigList))
for l in range(len(layers)):
    sns.scatterplot(x=slopes, y=eigList[l], s=2.5, linewidth=0, color=colors[l], label=f"layers: {l + 2}")
    # plt.plot(slopes, (slopes+0.5) ** 2, linewidth=5, color=colors[l])
plt.legend(loc="lower right", numpoints=1, markerscale=3)
plt.yscale('log')
plt.savefig(f"figure/leaky_relu_initial_eigenvalue.pdf", format="pdf")